# 08b · dense 앵커 교체 진단 — "BOILER 앵커가 G_cos를 망쳤나?"

**가설(H4 후속)**: dense `G_cos`가 `tfidf_G`와 −0.06으로 안 붙는 건, G 앵커 6개 중
4개(G1 이사회구성·G2 이사회활동·G3 주주권리·G5 감사기구)가 **모든 보고서에 깔린 BOILER**라
회사 변별이 안 되기 때문이다. 변별력 있는 **DRIVER(G4 윤리경영·G6 법규준수)만**으로 다시 재면
`tfidf_G`·`g_grade`와의 상관이 **−0.06 → 양(+)으로 올라갈** 것이다.

## 실행 전제
이 노트북은 **`07b_dense_esg_v2_colab.ipynb`를 위→아래 끝까지(최소 Step 6.1까지) 실행한
같은 런타임**에 이어 붙여 돌린다. 아래 객체를 그 노트북에서 그대로 재사용한다:
`S_w`(화이트닝 문장임베딩) · `whiten_abtt` · `anchor_texts/anchor_dim/anchor_sub` · `A`(화이트닝 앵커)
· `model`(KURE-v1) · `pool` · `df`(문장 parquet) · `doc_keys` · `grades` · `report_len` · `PROJECT/DATA`.

**무거운 재인코딩 없음** — 기존 문장임베딩 `S_w`를 그대로 쓰고, 새 앵커(이미 6개 다 ANCHORS에 있음)는
기존 `A`에서 인덱스로 골라낼 뿐이다. (혹시 `A`가 없으면 셀 1이 KURE로 6문장만 즉석 인코딩 — CPU도 수초.)

결과 CSV: `data/processed/08b_dense_anchor_*.csv` → 끝에 요약 출력. 그 숫자를 Claude에게 주면 해석.

> **⚠ 주의 (이 노트북이 고친 함정)**: 07b Step 11 에서 변수 `A` 가 **통제변수 DataFrame(381×42)** 으로\n> 덮어쓰여진다. 그래서 화이트닝 앵커를 `A` 로 쓰면 `S_w @ A.T` 가 shape mismatch 로 터진다.\n> → 셀 1 이 `A_raw`+`whiten_abtt` 로 화이트닝 앵커를 **`Aw`** 라는 새 이름으로 복원해 쓴다.

In [ ]:
# ── 셀 1 · 전제 객체 확보 + 화이트닝 앵커(Aw) 복원 ──────────────────────────
# [중요] 07b Step11 에서 변수 `A` 가 통제변수 DataFrame(381×42)으로 덮어쓰여진다.
#   → 화이트닝 앵커는 `A` 가 아니라, Step5.1 의 `A_raw`+`whiten_abtt` 로 다시 만든다(=Aw).
import numpy as np, pandas as pd
from pathlib import Path
from scipy.stats import spearmanr

try:
    PROJECT, DATA
except NameError:
    PROJECT = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA = PROJECT / "data" / "processed"
print("PROJECT =", PROJECT)

# ── 화이트닝 문장임베딩 S_w 확보 (np.ndarray 보장) ──
assert "S_w" in globals(), "S_w 없음 — 07b Step 5.2(S_w = whiten_abtt(E)) 까지 선행 필수."
S_w = np.asarray(S_w, dtype=np.float32)

# ── 화이트닝 앵커 Aw 복원 ──
#  우선순위: ① A_raw+whiten_abtt 로 재생성(가장 안전)  ② 메모리의 A 가 진짜 앵커면 그대로
def _looks_like_anchor(x):
    return isinstance(x, np.ndarray) and x.ndim == 2 and x.shape[1] == S_w.shape[1]

if "A_raw" in globals() and "whiten_abtt" in globals():
    Aw = whiten_abtt(np.asarray(A_raw, dtype=np.float32))
    print("Aw = whiten_abtt(A_raw) 로 재생성")
elif "A" in globals() and _looks_like_anchor(A):
    Aw = np.asarray(A, dtype=np.float32)
    print("기존 A 가 화이트닝 앵커로 보여 그대로 사용")
else:
    # 최후: KURE 로 앵커 즉석 인코딩(42문장, 수초) — anchor_texts·whiten 파라미터 필요
    assert "anchor_texts" in globals(), "anchor_texts 없음 — 07b Step 5.1 선행 필수."
    from sentence_transformers import SentenceTransformer
    MODEL_NAME = globals().get("MODEL_NAME", "nlpai-lab/KURE-v1")
    _m = globals().get("model") or SentenceTransformer(MODEL_NAME)
    A_raw = _m.encode(anchor_texts, normalize_embeddings=True, convert_to_numpy=True).astype(np.float32)
    if "whiten_abtt" in globals():
        Aw = whiten_abtt(A_raw)
    else:  # whiten 파라미터 npz 직접 로드
        wz = np.load(globals().get("WHITEN_NPZ", DATA / "07_v2_whiten.npz"), allow_pickle=True)
        mu_w, std_w, top_w = wz["mu"], wz["std"], wz["abtt_top"]
        def _l2n(M): n = np.linalg.norm(M, axis=1, keepdims=True); n[n == 0] = 1; return M / n
        Xs = (A_raw - mu_w) / std_w
        Aw = _l2n(Xs - (Xs @ top_w) @ top_w.T)
    print("Aw = KURE 즉석 인코딩+화이트닝 으로 생성")

print(f"S_w shape={S_w.shape} | Aw shape={Aw.shape}")
assert Aw.shape[1] == S_w.shape[1], f"차원 불일치 S_w {S_w.shape} vs Aw {Aw.shape}"

# ── 앵커 메타 점검 ──
for k in ["anchor_dim", "anchor_sub", "anchor_texts", "df", "doc_keys", "grades", "report_len"]:
    assert k in globals(), f"{k} 없음 — 07b 를 Step 6.1 까지 위→아래 실행 후 재시도."
anchor_dim = np.asarray(anchor_dim); anchor_sub = np.asarray(anchor_sub)
assert len(anchor_dim) == Aw.shape[0], f"앵커수 불일치: meta {len(anchor_dim)} vs Aw {Aw.shape[0]}"
print("앵커 차원 분포:", {d: int((anchor_dim == d).sum()) for d in ["E", "S", "G"]})
print("G 하위앵커:", [s for s, d in zip(anchor_sub, anchor_dim) if d == "G"])

In [ ]:
# ── 셀 2 · DRIVER / BOILER 앵커 인덱스 분리 ─────────────────────────────────
# G4 윤리경영 · G6 지배구조 법규 = DRIVER(선택적 변별어)
# G1 이사회구성 · G2 이사회활동 · G3 주주권리 · G5 감사기구 = BOILER(의무·보일러플레이트)
DRIVER_SUBS = ("G4", "G6")
BOILER_SUBS = ("G1", "G2", "G3", "G5")

g_mask = (anchor_dim == "G")
def _subset(prefixes):
    return np.array([i for i in range(len(anchor_sub))
                     if anchor_dim[i] == "G" and str(anchor_sub[i]).startswith(prefixes)])
idx_all    = np.where(g_mask)[0]
idx_driver = _subset(DRIVER_SUBS)
idx_boiler = _subset(BOILER_SUBS)

print("G 전체 앵커 :", [anchor_sub[i] for i in idx_all], f"(n={len(idx_all)})")
print("DRIVER 앵커 :", [anchor_sub[i] for i in idx_driver], f"(n={len(idx_driver)})")
print("BOILER 앵커 :", [anchor_sub[i] for i in idx_boiler], f"(n={len(idx_boiler)})")
assert len(idx_driver) >= 1 and len(idx_boiler) >= 1, "앵커 분리 실패 — anchor_sub 접두어 확인"
for nm, txts in [("DRIVER", idx_driver), ("BOILER", idx_boiler)]:
    print(f"\n[{nm} 앵커 문장]")
    for i in txts: print("   •", anchor_texts[i])

In [ ]:
# ── 셀 3 · 3가지 G 앵커셋으로 문서 G점수 재계산 (동일 파이프라인) ──────────────
# 07b Step6.1 과 똑같이: 문장→앵커 max → 풀링. docmean·max·topk 모두 산출.
# S_w = 화이트닝 문장임베딩 (N, dim), Aw = 화이트닝 앵커 (n_anc, dim)  ← 셀1에서 복원한 Aw 사용
sims_all = (S_w @ Aw.T).astype(np.float32)           # (N, n_anc)
print("sims_all shape:", sims_all.shape)

doc_sizes = df["sentences"].map(len).to_numpy()
offs = np.concatenate([[0], np.cumsum(doc_sizes)])
ndoc = len(doc_sizes)
assert offs[-1] == sims_all.shape[0], f"문장수 불일치: doc합 {offs[-1]} vs S_w {sims_all.shape[0]} — S_w 가 이 df 의 explode 순서와 같은지 확인."

def doc_scores(anchor_idx, pool_kind="docmean", topk=20):
    """문장→(주어진 앵커들 max)→문서 풀링. anchor_idx 가 가리키는 앵커만 사용."""
    sent_sim = sims_all[:, anchor_idx].max(axis=1)   # (N,) 문장별 = 해당 앵커군 최댓값
    out = np.empty(ndoc, dtype=np.float64)
    for d in range(ndoc):
        s = sent_sim[offs[d]:offs[d + 1]]
        if len(s) == 0:
            out[d] = np.nan; continue
        if pool_kind == "docmean": out[d] = s.mean()
        elif pool_kind == "max":   out[d] = s.max()
        elif pool_kind == "topk":  k = min(topk, len(s)); out[d] = np.sort(s)[-k:].mean()
        else: raise ValueError(pool_kind)
    return out

SETS = {"G_all": idx_all, "G_driver": idx_driver, "G_boiler": idx_boiler}
POOLS = ["docmean", "max", "topk"]
scores = {f"{name}__{pk}": doc_scores(idx, pk) for name, idx in SETS.items() for pk in POOLS}
sc = pd.DataFrame(scores)
sc["doc_key"] = list(doc_keys)
sc["report_len"] = np.asarray(report_len)
sc["g_grade"] = np.asarray(grades["G"])
print("문서 G점수(3앵커셋 × 3풀링) 산출:", sc.shape)
sc.head(3)

In [ ]:
# ── 셀 4 · tfidf_G 머지 ───────────────────────────────────────────────────
tf = pd.read_parquet(DATA / "05_tfidf_seed_features.parquet")
tf["doc_key"] = tf["stock_code"].astype(str).str.zfill(6) + "_" + tf["fiscal_year"].astype(int).astype(str)
sc = sc.merge(tf[["doc_key", "tfidf_G", "tfidf_E", "tfidf_S"]], on="doc_key", how="left")
miss = sc["tfidf_G"].isna().sum()
print(f"tfidf_G 머지 — 결측 {miss} / {len(sc)}")
assert miss == 0, "tfidf_G 결측 — doc_key 규칙(zfill6_연도) 확인"
sc.head(3)

In [ ]:
# ── 셀 5 · ★핵심★ 수렴타당도 표: 각 앵커셋 G점수 ↔ {tfidf_G, g_grade, report_len} ──
rows = []
for name in SETS:
    for pk in POOLS:
        col = f"{name}__{pk}"
        m = sc[[col, "tfidf_G", "g_grade", "report_len"]].dropna()
        rows.append({
            "anchor_set": name, "pool": pk,
            "rho_tfidf_G":   round(spearmanr(m[col], m["tfidf_G"]).statistic, 3),
            "rho_g_grade":   round(spearmanr(m[col], m["g_grade"]).statistic, 3),
            "rho_report_len": round(spearmanr(m[col], m["report_len"]).statistic, 3),
        })
conv = pd.DataFrame(rows)
conv.to_csv(DATA / "08b_dense_anchor_convergence.csv", index=False)
print("=" * 78)
print("수렴타당도: G 앵커셋별 점수가 tfidf_G / g_grade / report_len 과 얼마나 붙나")
print("  (가설: driver 가 all·boiler 보다 tfidf_G·g_grade 와 더 양으로 붙어야 함)")
print("=" * 78)
print(conv.to_string(index=False))

In [ ]:
# ── 셀 6 · driver vs boiler 점수 자체 상관 + g_grade 회귀(완전통제 없이 단순 비교) ──
# (1) driver 와 boiler 점수가 서로 얼마나 다른가 (낮을수록 '다른 걸 잰다')
print("▶ DRIVER vs BOILER 점수 상관 (풀링별):")
for pk in POOLS:
    r = spearmanr(sc[f"G_driver__{pk}"], sc[f"G_boiler__{pk}"]).statistic
    print(f"   {pk:8s} ρ(driver, boiler) = {r:+.3f}")

# (2) 채택 풀링(docmean) 기준, driver·boiler·all 의 g_grade 예측 부호/세기 한눈에
print("\n▶ docmean 기준 g_grade 예측 (raw ρ):")
for name in SETS:
    r = spearmanr(sc[f"{name}__docmean"], sc["g_grade"]).statistic
    rt = spearmanr(sc[f"{name}__docmean"], sc["tfidf_G"]).statistic
    print(f"   {name:9s}  ρ_g_grade={r:+.3f}   ρ_tfidf_G={rt:+.3f}")

# (3) 결론 자동 판정
rd = spearmanr(sc["G_driver__docmean"], sc["tfidf_G"]).statistic
ra = spearmanr(sc["G_all__docmean"], sc["tfidf_G"]).statistic
rb = spearmanr(sc["G_boiler__docmean"], sc["tfidf_G"]).statistic
print("\n" + "=" * 78)
print(f"[자동판정] tfidf_G 와의 상관:  driver={rd:+.3f}  all={ra:+.3f}  boiler={rb:+.3f}")
if rd > ra and rd > rb:
    print("  → ✅ 가설 지지: DRIVER 앵커만 쓰면 tfidf_G 와 더 잘 붙는다.")
    print("     = G_cos 가 −0.06 으로 죽은 건 BOILER 앵커가 신호를 희석했기 때문.")
else:
    print("  → ❌ 가설 기각/불명확: BOILER 제거가 tfidf_G 상관을 올리지 못함.")
    print("     = 실패 원인이 '앵커 구성'이 아니라 더 깊은 곳(임베딩이 변별어 자체를 못 가름)일 수 있음.")
print("=" * 78)

In [ ]:
# ── 셀 7 · (선택) 완전통제 부분상관까지 — 통제변수 테이블이 있으면 ──────────────
# 길이·규모·업종·연도 통제 후에도 driver 가 살아남나?
# 07b Step11 에서 `A` 가 통제 테이블(381×42, sector_/year_ 더미 포함)로 덮였으므로 그걸 우선 활용.
try:
    import statsmodels.api as sm
    AT = None
    # ① 메모리의 A 가 통제 테이블(DataFrame, sector_ 컬럼 보유)이면 사용
    if "A" in globals() and isinstance(A, pd.DataFrame) and any(str(c).startswith("sector_") for c in A.columns):
        AT = A.copy()
    # ② 아니면 CSV 로드
    if AT is None and (DATA / "07_v2_analysis_table.csv").exists():
        AT = pd.read_csv(DATA / "07_v2_analysis_table.csv", dtype={"stock_code": "string"})
    if AT is not None:
        if "doc_key" not in AT.columns:
            if "stock_code" in AT.columns and "fiscal_year" in AT.columns:
                sc_str = AT["stock_code"].astype("string").str.replace(r"\.0$", "", regex=True).str.zfill(6)
                AT["doc_key"] = sc_str + "_" + AT["fiscal_year"].astype(int).astype(str)
        sector_cols = sorted(c for c in AT.columns if str(c).startswith("sector_"))
        year_cols   = sorted(c for c in AT.columns if str(c).startswith("year_"))
        CTRL = [c for c in ["log_core_doc_len_z", "log_assets_z"] if c in AT.columns] + sector_cols + year_cols
        if "doc_key" in AT.columns and CTRL:
            zc = lambda s: (s - s.mean()) / s.std(ddof=0)
            def partial_resid(y, X):
                X1 = sm.add_constant(X.astype(float), has_constant="add")
                return y.values - X1.values @ np.linalg.lstsq(X1.values, y.values, rcond=None)[0]
            mrg = sc.merge(AT[["doc_key"] + CTRL].drop_duplicates("doc_key"), on="doc_key", how="inner")
            print(f"완전통제 부분ρ (통제 {len(CTRL)}변수, n={len(mrg)}):")
            for name in SETS:
                col = f"{name}__docmean"
                mm = mrg[[col, "g_grade"] + CTRL].dropna()
                X = mm[CTRL]
                rx = partial_resid(zc(mm[col]), X); ry = partial_resid(mm["g_grade"].astype(float), X)
                print(f"   {name:9s}  partial_ρ(g_grade | 통제) = {spearmanr(rx, ry).statistic:+.3f}")
        else:
            print("[skip] doc_key 또는 통제컬럼 없음 → 셀 5 raw 결과로 충분.")
    else:
        print("[skip] 통제변수 테이블 없음 → 셀 5 raw 결과로 충분.")
except Exception as e:
    print("[skip] 완전통제 부분상관 생략:", repr(e))

## 결과를 Claude 에게 줄 때
**셀 5 표(`08b_dense_anchor_convergence.csv`)** 와 **셀 6 자동판정 출력**을 복사해서 주면 된다.

핵심으로 볼 것:
- `G_driver` 의 `rho_tfidf_G` 가 `G_all`(≈ 기존 −0.06 재현)·`G_boiler` 보다 **확실히 큰 양수**면 → 가설 지지.
- `G_boiler` 의 `rho_g_grade` 가 0 근처거나 음수면 → 보일러플레이트가 변별 무용(H2 와 일치).
- driver 도 여전히 0 근처면 → 실패 원인이 앵커가 아니라 **임베딩이 희소 변별어를 의미공간에서 분리 못 함**(더 깊은 구조적 한계).